In [1]:
!python --version

Python 3.10.15


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import tkinter as tk
from threading import Thread
from PIL import Image, ImageTk
from tkinter import messagebox

# Global variables
start_counting = False
counter = 0
stage = None
rep_goal = 12  # Default goal
running = True  # To safely terminate threads and resources
cap = None  # Global video capture object
frame = None  # Shared frame between threads

# Define the angle calculation function
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)
    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)
    if angle > 180.0:
        angle = 360 - angle
    return angle

# Function to handle Mediapipe Pose detection
def mediapipe_thread():
    global cap, frame, start_counting, counter, stage, running

    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils

    # Start video capture
    cap = cv2.VideoCapture(1)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)  # Reduce buffering for real-time processing

    # Initialize Mediapipe Pose
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while running:
            ret, raw_frame = cap.read()
            if not ret:
                break

            # Flip the frame to mirror it
            raw_frame = cv2.flip(raw_frame, 1)
            
            # Resize for faster processing
            small_frame = cv2.resize(raw_frame, (480, 360))
            image = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = pose.process(image)
            image.flags.writeable = True
            small_frame = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            # Bicep curl logic
            if start_counting and results.pose_landmarks:
                try:
                    landmarks = results.pose_landmarks.landmark

                    # Get coordinates
                    shoulder = [landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].x,
                                landmarks[mp_pose.PoseLandmark.RIGHT_SHOULDER.value].y]
                    elbow = [landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].x,
                             landmarks[mp_pose.PoseLandmark.RIGHT_ELBOW.value].y]
                    wrist = [landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].x,
                             landmarks[mp_pose.PoseLandmark.RIGHT_WRIST.value].y]

                    # Calculate angle
                    angle = calculate_angle(shoulder, elbow, wrist)

                    # Visualize angle
                    cv2.putText(small_frame, str(int(angle)),
                                tuple(np.multiply(elbow, [640, 360]).astype(int)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2, cv2.LINE_AA)

                    # Rep counting logic
                    if angle > 160:
                        stage = "down"
                    if angle < 50 and stage == "down":
                        stage = "up"
                        counter += 1
                        print(f"Rep Count: {counter}")

                        if counter >= rep_goal:
                            messagebox.showinfo("Set Complete", f"Congratulations! You've completed {rep_goal} reps!")
                            start_counting = False
                except Exception as e:
                    pass

            # Draw counter and goal
            cv2.rectangle(small_frame, (0, 0), (300, 100), (245, 117, 16), -1)
            cv2.putText(small_frame, f'REPS: {counter}/{rep_goal}', (10, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2, cv2.LINE_AA)

            frame = small_frame

        cap.release()

# Function to update the GUI with frames
def update_gui():
    global frame, running

    if frame is not None:
        # Convert frame to ImageTk format for GUI
        img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = Image.fromarray(img)
        imgtk = ImageTk.PhotoImage(image=img)

        # Update the GUI label with the new frame
        webcam_label.imgtk = imgtk
        webcam_label.configure(image=imgtk)

    if running:
        webcam_label.after(10, update_gui)  # Update every 10ms for a smoother GUI

# Function to handle the "Start Counting" button click
def start_button_click():
    global start_counting, counter
    counter = 0  # Reset counter
    start_counting = True

# Function to handle the "Stop Counting" button click
def stop_button_click():
    global start_counting, counter, rep_goal
    if start_counting:
        if counter < rep_goal:
            messagebox.showinfo("Set Stopped", f"You stopped at {counter} reps. Your goal was {rep_goal} reps.")
        start_counting = False

# Function to set a new rep goal
def set_goal():
    global rep_goal
    try:
        new_goal = int(goal_entry.get())
        if new_goal > 0:
            rep_goal = new_goal
            messagebox.showinfo("Goal Updated", f"Your new goal is {rep_goal} reps!")
        else:
            messagebox.showerror("Invalid Input", "Please enter a positive number.")
    except ValueError:
        messagebox.showerror("Invalid Input", "Please enter a valid number.")

# GUI setup
root = tk.Tk()
root.title("Bicep Curl Counter")

# Webcam display area
webcam_label = tk.Label(root)
webcam_label.pack()

# Buttons frame
button_frame = tk.Frame(root)
button_frame.pack(pady=10)

# Create Start Button
start_button = tk.Button(button_frame, text="Start Counting", command=start_button_click, font=("Arial", 16), bg="green", fg="white")
start_button.pack(side=tk.LEFT, padx=10)

# Create Stop Button
stop_button = tk.Button(button_frame, text="Stop Counting", command=stop_button_click, font=("Arial", 16), bg="red", fg="white")
stop_button.pack(side=tk.LEFT, padx=10)

# Goal input frame
goal_frame = tk.Frame(root)
goal_frame.pack(pady=10)

goal_label = tk.Label(goal_frame, text="Set Rep Goal:", font=("Arial", 14))
goal_label.pack(side=tk.LEFT)

goal_entry = tk.Entry(goal_frame, font=("Arial", 14), width=5)
goal_entry.pack(side=tk.LEFT, padx=5)

set_goal_button = tk.Button(goal_frame, text="Set Goal", command=set_goal, font=("Arial", 14), bg="blue", fg="white")
set_goal_button.pack(side=tk.LEFT)

# Run Mediapipe processing in a separate thread
mediapipe_thread = Thread(target=mediapipe_thread)
mediapipe_thread.daemon = True
mediapipe_thread.start()

# Start GUI update loop
update_gui()

# Run GUI event loop
root.mainloop()

# Safely stop the Mediapipe thread
running = False
mediapipe_thread.join()


Rep Count: 1
Rep Count: 2
Rep Count: 3
Rep Count: 4
Rep Count: 5
Rep Count: 6
Rep Count: 7
Rep Count: 8
Rep Count: 9
Rep Count: 10
Rep Count: 11
Rep Count: 12
Rep Count: 1
Rep Count: 2
Rep Count: 3
Rep Count: 4
Rep Count: 5
Rep Count: 6
Rep Count: 7
Rep Count: 8
Rep Count: 9
Rep Count: 10
Rep Count: 11
